In [ ]:
!pip install -q sentence-transformers datasets torch accelerate

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
import torch

In [ ]:
dataset = load_dataset('Abeshith/research-embedding-dataset')
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'source_dataset', 'query', 'positive', 'negatives', 'query_id', 'positive_id', 'negative_ids', 'task_type', 'domain'],
        num_rows: 1210
    })
    validation: Dataset({
        features: ['id', 'source_dataset', 'query', 'positive', 'negatives', 'query_id', 'positive_id', 'negative_ids', 'task_type', 'domain'],
        num_rows: 24
    })
})


In [ ]:
dataset['train'][0]

{'id': '86ef4b3968279ffb992c429a14086287',
 'source_dataset': 'mteb/scifact',
 'query': 'Energy balance requires hypothalamic glutamate neurotransmission.',
 'positive': 'Synaptic glutamate release by ventromedial hypothalamic neurons is part of the neurocircuitry that prevents hypoglycemia.. The importance of neuropeptides in the hypothalamus has been experimentally established. Due to difficulties in assessing function in vivo, the roles of the fast-acting neurotransmitters glutamate and GABA are largely unknown. Synaptic vesicular transporters (VGLUTs for glutamate and VGAT for GABA) are required for vesicular uptake and, consequently, synaptic release of neurotransmitters. Ventromedial hypothalamic (VMH) neurons are predominantly glutamatergic and express VGLUT2. To evaluate the role of glutamate release from VMH neurons, we generated mice lacking VGLUT2 selectively in SF1 neurons (a major subset of VMH neurons). These mice have hypoglycemia during fasting secondary to impaired fas

In [ ]:
train_dataset = dataset['train'].shuffle(seed=42).map(lambda x: {
    'query': x['query'],
    'positive': x['positive'],
})

Map:   0%|          | 0/1210 [00:00<?, ? examples/s]

In [ ]:
eval_dataset = dataset['validation']

In [ ]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
loss = MultipleNegativesRankingLoss(model)

In [ ]:
args = SentenceTransformerTrainingArguments(
    output_dir='./finetuned-research-embedding',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
)

In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

In [ ]:
trainer.train()

dataset = dataset.select_columns(['anchor', 'positive', 'negative'])


Epoch,Training Loss,Validation Loss
1,0.000017,0.000001
2,0.000012,0.000001
3,0.000004,0.000001


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TrainOutput(global_step=456, training_loss=0.15750344176041453, metrics={'train_runtime': 416.8006, 'train_samples_per_second': 8.709, 'train_steps_per_second': 1.094, 'total_flos': 0.0, 'train_loss': 0.15750344176041453, 'epoch': 3.0})

In [ ]:
args.num_train_epochs = 10
trainer.train(resume_from_checkpoint=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
4,0.000003,0.000000
5,0.000002,0.000000
6,0.000001,0.000000
7,0.000002,0.000000
8,0.000001,0.000000
9,0.000001,0.000000
10,0.000001,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

TrainOutput(global_step=1520, training_loss=1.1626042817768299e-06, metrics={'train_runtime': 977.4935, 'train_samples_per_second': 12.379, 'train_steps_per_second': 1.555, 'total_flos': 0.0, 'train_loss': 1.1626042817768299e-06, 'epoch': 10.0})

In [ ]:
trainer.save_model('./finetuned-research-embedding-final')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
base_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
finetuned_model = SentenceTransformer('./finetuned-research-embedding-final')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import SentenceTransformer, util

base_similarities = []
finetuned_similarities = []

for i in range(min(10, len(eval_dataset))):  # First 10 samples
    sample = eval_dataset[i]
    query_emb_base = base_model.encode(sample['query'], convert_to_tensor=True)
    pos_emb_base = base_model.encode(sample['positive'], convert_to_tensor=True)
    base_sim = util.cos_sim(query_emb_base, pos_emb_base)[0][0].item()

    query_emb_ft = finetuned_model.encode(sample['query'], convert_to_tensor=True)
    pos_emb_ft = finetuned_model.encode(sample['positive'], convert_to_tensor=True)
    ft_sim = util.cos_sim(query_emb_ft, pos_emb_ft)[0][0].item()

    base_similarities.append(base_sim)
    finetuned_similarities.append(ft_sim)

    print(f"Sample {i+1:2d}: Base={base_sim:.4f} | Finetuned={ft_sim:.4f} | Δ={ft_sim-base_sim:+.4f}")

Sample  1: Base=0.8143 | Finetuned=0.8242 | Δ=+0.0099
Sample  2: Base=0.3958 | Finetuned=0.3839 | Δ=-0.0119
Sample  3: Base=0.3679 | Finetuned=0.3626 | Δ=-0.0053
Sample  4: Base=0.3909 | Finetuned=0.4090 | Δ=+0.0180
Sample  5: Base=0.5675 | Finetuned=0.5481 | Δ=-0.0195
Sample  6: Base=0.3717 | Finetuned=0.4104 | Δ=+0.0387
Sample  7: Base=0.5119 | Finetuned=0.5274 | Δ=+0.0154
Sample  8: Base=0.8704 | Finetuned=0.8653 | Δ=-0.0051
Sample  9: Base=0.5099 | Finetuned=0.5273 | Δ=+0.0175
Sample 10: Base=0.5155 | Finetuned=0.5377 | Δ=+0.0222


In [ ]:
import numpy as np

print(f"\nAvg Cosine Sim - Base: {np.mean(base_similarities):.4f} | Finetuned: {np.mean(finetuned_similarities):.4f}")
print(f"Improvement: {np.mean(finetuned_similarities) - np.mean(base_similarities):+.4f}")


Avg Cosine Sim - Base: 0.5316 | Finetuned: 0.5396
Improvement: +0.0080


In [ ]:
train_queries = dataset['train']['query'][100:110]
train_docs = dataset['train']['positive'][100:150]

base_scores_all = []
ft_scores_all = []

# Test each query against full doc corpus
for i, query in enumerate(train_queries):
    print(f"Query {i+1}: {query[:80]}...")

    # Encode query + all docs
    q_base = base_model.encode([query])
    q_ft = finetuned_model.encode([query])
    docs_base = base_model.encode(train_docs, show_progress_bar=False)
    docs_ft = finetuned_model.encode(train_docs, show_progress_bar=False)

    # Cosine similarity scores
    base_scores = util.cos_sim(q_base, docs_base)[0]
    ft_scores = util.cos_sim(q_ft, docs_ft)[0]

    # Store top scores
    base_scores_all.extend(base_scores.cpu().numpy())
    ft_scores_all.extend(ft_scores.cpu().numpy())

    # Show top-3 for this query
    base_top3 = base_scores.topk(3).values.cpu().numpy()
    ft_top3 = ft_scores.topk(3).values.cpu().numpy()

    print(f"  Base Top-3:  [{base_top3[0]:.4f}, {base_top3[1]:.4f}, {base_top3[2]:.4f}]")
    print(f"  Finetuned Top-3: [{ft_top3[0]:.4f}, {ft_top3[1]:.4f}, {ft_top3[2]:.4f}]")
    print()

Query 1: what are best practices in hospitals and at home in maintaining quarantine?...
  Base Top-3:  [0.4116, 0.4040, 0.3975]
  Finetuned Top-3: [0.4033, 0.3949, 0.3935]

Query 2: what are the best masks for preventing infection by Covid-19?...
  Base Top-3:  [0.5270, 0.3327, 0.3300]
  Finetuned Top-3: [0.5380, 0.3171, 0.3111]

Query 3: Satellite cell dysfunction is a key factor in sarcopenia development....
  Base Top-3:  [0.6642, 0.4285, 0.4133]
  Finetuned Top-3: [0.6710, 0.4644, 0.4494]

Query 4: Ambulatory blood pressure monitoring is inaccurate at diagnosing hypertension....
  Base Top-3:  [0.6592, 0.3892, 0.2564]
  Finetuned Top-3: [0.6717, 0.3763, 0.2459]

Query 5: What is the mechanism of inflammatory response and pathogenesis of COVID-19 case...
  Base Top-3:  [0.5676, 0.4675, 0.4675]
  Finetuned Top-3: [0.5850, 0.4881, 0.4881]

Query 6: How does the coronavirus differ from seasonal flu?...
  Base Top-3:  [0.4442, 0.3833, 0.3806]
  Finetuned Top-3: [0.4222, 0.3622, 0.3418]


In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('./finetuned-research-embedding-final')
model.push_to_hub("Abeshith/research-embedding-finetuned")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ylbuxxe/model.safetensors:   2%|2         | 10.7MB /  438MB            

'https://huggingface.co/Abeshith/research-embedding-finetuned/commit/1e673b828aad977e99e4a5eedb0c268bfb922b13'